## Subcortical Volume — Prodromal Subgroup Analysis

Compares **RBD** and **Hyposmia** subgroups against **Healthy Controls** using NiSpace.

- Contrasts: RBD vs HC · Hyposmia vs HC
- Phenotype: Subcortical volume (Aseg, 14 bilateral structures)
- Subgroup IDs: RBD = 5.0, Hyposmia = 6.0, HC = 2.0
- Same reference maps, covariates, and helper functions as `volume_subcortical_shi.ipynb`

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import statsmodels.api as sm

np.random.seed(42)

from nispace.datasets import fetch_reference
from nispace.workflows import group_comparison

In [ ]:
DATA_PATH  = Path("../../data/df1.csv")
PARCELLATION = "Aseg"

# Exact normalized region names for the 7 bilateral structures in the NiSpace Aseg reference.
# Uses exact matching (not substring) to avoid "thalamus" hitting "thalamusproper"
# or "accumbens" hitting "accumbensarea".
ASEG_BASES = {
    "thalamus", "caudate", "putamen", "pallidum",
    "hippocampus", "amygdala", "accumbensarea",
}

DEMO_COLS = ["PATNO", "CONCOHORT", "age", "SEX", "agediag", "subgroup", "PRIMDIAG"]

SUBGROUP_TO_ID = {"Healthy Control": 2.0, "RBD": 5.0, "Hyposmia": 6.0}

CONTRASTS = {
    "RBD vs HC": (5.0, 2.0),
    "Hyposmia vs HC": (6.0, 2.0),
}

GROUP_LABELS = {5.0: "RBD", 6.0: "Hyposmia", 2.0: "HC"}

SELECTED_REFERENCE_MAPS = [
    "target-mGluR5_tracer-abp688_n-73_dx-hc_pub-smart2019",
    "target-NMDA_tracer-ge179_n-29_dx-hc_pub-galovic2021",
    "target-GABAa_tracer-flumazenil_n-6_dx-hc_pub-dukart2018",
    "target-FDOPA_tracer-fluorodopa_n-12_dx-hc_pub-garciagomez2018",
    "target-D1_tracer-sch23390_n-13_dx-hc_pub-kaller2017",
    "target-D23_tracer-flb457_n-55_dx-hc_pub-sandiego2015",
    "target-DAT_tracer-fpcit_n-174_dx-hc_pub-dukart2018",
    "target-5HT1a_tracer-way100635_n-35_dx-hc_pub-savli2012",
    "target-5HT1b_tracer-p943_n-23_dx-hc_pub-savli2012",
    "target-5HT2a_tracer-altanserin_n-19_dx-hc_pub-savli2012",
    "target-5HT4_tracer-sb207145_n-59_dx-hc_pub-beliveau2017",
    "target-5HTT_tracer-dasb_n-18_dx-hc_pub-savli2012",
    "target-NET_tracer-mrb_n-10_dx-hc_pub-hesse2017",
    "target-VAChT_tracer-feobv_n-18_dx-hc_pub-aghourian2017",
]

N_PERM = 10000

In [ ]:
# ── Helper functions (identical to volume_subcortical_shi.ipynb) ──────────────

def _norm(s):
    s = str(s).strip().lower()
    s = s.replace("left", "lh").replace("right", "rh")
    s = s.replace("ctx-lh-", "lh_").replace("ctx-rh-", "rh_")
    s = s.replace("aparc_", "").replace("aseg_", "")
    s = re.sub(r"[\s\-\./ \+]+", "_", s)
    s = re.sub(r"[^a-z0-9_]", "", s)
    s = re.sub(r"_+", "_", s).strip("_")
    s = re.sub(r"(_)?volume$", "", s)
    s = re.sub(r"(_)?vol$", "", s)
    s = s.replace("thalamus_proper", "thalamusproper")
    s = s.replace("accumbens_area", "accumbensarea")
    s = s.replace("ventral_dc", "ventraldc")
    return s

def get_aseg_volume_columns(df):
    cols = []
    for c in df.columns:
        cn = _norm(c)
        # Strip hemisphere prefix, then check EXACT match against ASEG_BASES.
        # Exact matching prevents "thalamus" from also matching "thalamusproper"
        # and "accumbens" from matching "accumbensarea" (both are substring matches
        # that would otherwise inflate the column count from 14 to 16+).
        m = re.match(r"^(lh|rh)_([a-z0-9]+)$", cn)
        if not m:
            continue
        region = m.group(2)
        if region in ASEG_BASES:
            cols.append(c)
    return cols

def subject_col_to_key(col):
    cn = _norm(col)
    m = re.match(r"^(lh|rh)_([a-z0-9_]+)$", cn)
    if m:
        hemi, region = m.groups()
        return f"{hemi}_{region.replace('_','')}"
    return cn

def reference_col_to_key(col):
    s = str(col)
    m1 = re.match(r"^hemi-([LR])_lab-(.+)$", s)
    if m1:
        hemi, region = m1.groups()
        hemi = "lh" if hemi == "L" else "rh"
        return f"{hemi}_{_norm(region).replace('_','')}"
    sn = _norm(s)
    m2 = re.match(r"^(lh|rh)_(.+)$", sn)
    if m2:
        hemi, region = m2.groups()
        return f"{hemi}_{region.replace('_','')}"
    return sn.replace("_", "")

def build_subject_to_reference_mapping(subject_cols, reference_cols, verbose=True):
    subj_keys = {c: subject_col_to_key(c) for c in subject_cols}
    ref_keys  = {c: reference_col_to_key(c) for c in reference_cols}
    rev_ref = {}
    for ref_col, key in ref_keys.items():
        rev_ref.setdefault(key, []).append(ref_col)
    mapping = {}
    for sub_col, key in subj_keys.items():
        matches = rev_ref.get(key, [])
        if len(matches) == 1:
            mapping[sub_col] = matches[0]
    if verbose:
        print(f"Matched: {len(mapping)} / {len(subject_cols)}")
    return mapping

def pick_etiv_column(df):
    candidates = ["eTIV","EstimatedTotalIntraCranialVol","IntraCranialVol"]
    exact = [c for c in candidates if c in df.columns]
    if exact: return exact[0]
    for c in df.columns:
        if _norm(c) in {"etiv","estimatedtotalintracranialvol","intracranialvol"}: return c
    raise ValueError("Could not find eTIV column")

def prepare_brain_and_design(df):
    aseg_cols = get_aseg_volume_columns(df)
    etiv_col  = pick_etiv_column(df)
    print(f"Subcortical columns: {len(aseg_cols)}, eTIV: {etiv_col}")
    fs_col = "Field Strength"
    req_cols = ["PATNO","CONCOHORT","age","SEX",etiv_col] + ([fs_col] if fs_col in df.columns else [])
    df_sub = df[req_cols + aseg_cols].copy()
    Y = df_sub[aseg_cols].apply(pd.to_numeric, errors="coerce")
    Y.index = df_sub["PATNO"].astype(str)
    design = pd.DataFrame(index=Y.index)
    design["CONCOHORT"]      = pd.to_numeric(df_sub["CONCOHORT"], errors="coerce").values
    design["age"]            = pd.to_numeric(df_sub["age"],       errors="coerce").values
    design["SEX"]            = df_sub["SEX"].values
    design["eTIV"]           = pd.to_numeric(df_sub[etiv_col],    errors="coerce").values
    design["field_strength"] = pd.to_numeric(df_sub[fs_col],      errors="coerce").values
    return Y, design

def align_y_to_reference(Y, ref_df):
    mapping = build_subject_to_reference_mapping(Y.columns.tolist(), ref_df.columns.tolist())
    Y_aligned = Y.rename(columns=mapping).reindex(columns=ref_df.columns)
    print("Y aligned shape:", Y_aligned.shape)
    n_all_nan = Y_aligned.isna().all(axis=0).sum()
    if n_all_nan:
        print(f"  Note: {n_all_nan} reference parcels are all-NaN (e.g. VentralDC absent in data) — excluded from row filters")
    return Y_aligned

def run_parcelwise_ttest_vol(Y_aligned, design, g1, g2):
    mask = design["CONCOHORT"].astype(float).isin([g1, g2])
    y_sub = Y_aligned.loc[mask].copy()
    d_sub = design.loc[mask].copy()
    d_sub = d_sub.copy()
    d_sub["group01"] = d_sub["CONCOHORT"].astype(float).map({g1: 0, g2: 1})
    if d_sub["SEX"].dtype == object:
        d_sub["SEX"] = pd.Categorical(d_sub["SEX"]).codes
    d_sub["SEX"] = pd.to_numeric(d_sub["SEX"], errors="coerce")
    t_vals, p_vals, dfs = [], [], []
    for parcel in y_sub.columns:
        tmp = pd.DataFrame({
            "y": pd.to_numeric(y_sub[parcel], errors="coerce"),
            "group01": d_sub["group01"], "age": d_sub["age"],
            "SEX": d_sub["SEX"], "eTIV": d_sub["eTIV"],
            "field_strength": d_sub["field_strength"],
        }, index=y_sub.index).dropna()
        if tmp.shape[0] < 5 or tmp["group01"].nunique() < 2:
            t_vals.append(np.nan); p_vals.append(np.nan); dfs.append(np.nan)
            continue
        X = sm.add_constant(tmp[["group01","age","SEX","eTIV","field_strength"]])
        model = sm.OLS(tmp["y"], X).fit()
        t_vals.append(model.tvalues.get("group01", np.nan))
        p_vals.append(model.pvalues.get("group01", np.nan))
        dfs.append(model.df_resid)
    return pd.DataFrame(
        {"Tvalue": t_vals, "pvalue": p_vals, "df": dfs,
         "hemi": ["L" if "hemi-L" in str(c) else "R" for c in Y_aligned.columns]},
        index=Y_aligned.columns,
    )

def build_nispace_design_vol(d_sub, g1, g2):
    groups01 = d_sub["CONCOHORT"].astype(float).map({g1: 0, g2: 1}).astype(int)
    design_df = pd.DataFrame({
        "groups": groups01,
        "age":    pd.to_numeric(d_sub["age"],  errors="coerce"),
        "SEX":    d_sub["SEX"],
        "eTIV":   d_sub["eTIV"],
        "field_strength": d_sub["field_strength"],
    }, index=d_sub.index)
    if design_df["SEX"].dtype == object:
        design_df["SEX"] = pd.Categorical(design_df["SEX"]).codes
    design_df["SEX"] = pd.to_numeric(design_df["SEX"], errors="coerce")
    design_df["field_strength"] = design_df["field_strength"].map({1.5: 0, 3.0: 1})
    return design_df

def run_group_comparisons_vol(Y_aligned, design, ref_df, contrasts, labels, n_perm=10000):
    all_rows, outputs = [], {}
    for contrast_name, (g1, g2) in contrasts.items():
        mask = design["CONCOHORT"].astype(float).isin([g1, g2])
        y_sub = Y_aligned.loc[mask].copy()
        d_sub = design.loc[mask].copy()
        design_df = build_nispace_design_vol(d_sub, g1, g2)
        cov_cols = [c for c in ["groups","age","SEX","eTIV","field_strength"] if c in design_df.columns]
        # Skip parcels that are all-NaN (VentralDC absent in subject data) before row filter
        valid_parcels = y_sub.columns[~y_sub.isna().all(axis=0)]
        keep = ~(y_sub[valid_parcels].isna().any(axis=1) | design_df[cov_cols].isna().any(axis=1))
        y_sub     = y_sub.loc[keep]; design_df = design_df.loc[keep]
        d_sub     = d_sub.loc[keep]
        print(f"\nContrast: {contrast_name}")
        print("Counts:", d_sub["CONCOHORT"].astype(float).map(labels).value_counts().to_dict())
        np.random.seed(42)
        colocs, pvals, qvals, nsp = group_comparison(
            y=y_sub, x=ref_df, parcellation=PARCELLATION, design=design_df,
            comparison_method="hedges(a,b)", colocalization_method="spearman",
            n_perm=n_perm, n_proc=-1, verbose=True,
        )
        outputs[contrast_name] = {"colocs": colocs, "p": pvals, "q": qvals, "nsp": nsp}
        df_out = pd.DataFrame({"reference_map": ref_df.index, "rho": np.asarray(colocs).ravel(),
                               "p": np.asarray(pvals).ravel(), "q": np.asarray(qvals).ravel(),
                               "contrast": contrast_name})
        all_rows.append(df_out)
    return pd.concat(all_rows, ignore_index=True).sort_values(["contrast","q","p"]), outputs

In [ ]:
# ── Load and filter data ──────────────────────────────────────────────────────
df_raw = pd.read_csv(DATA_PATH, low_memory=False)
df = df_raw[df_raw["subgroup"].isin(SUBGROUP_TO_ID.keys())].copy()
df["CONCOHORT"] = df["subgroup"].map(SUBGROUP_TO_ID)
print("Subgroup counts:")
print(df["subgroup"].value_counts())
print("\nField Strength distribution:")
print(df["Field Strength"].value_counts(dropna=False))

In [ ]:
Y, design = prepare_brain_and_design(df)
print("Y shape:", Y.shape)
print("Design shape:", design.shape)

In [ ]:
df_reference = fetch_reference(
    "pet", collection="UniqueTracers", parcellation=PARCELLATION, print_references=True,
)
df_reference_selected = df_reference[
    df_reference.index.get_level_values("map").isin(SELECTED_REFERENCE_MAPS)
]
print("Selected reference shape:", df_reference_selected.shape)

In [ ]:
Y_aligned = align_y_to_reference(Y, df_reference_selected)
common_idx = Y_aligned.index.intersection(design.index)
Y_aligned = Y_aligned.loc[common_idx].copy()
design = design.loc[common_idx].copy()

In [ ]:
# ── Parcelwise t-tests ────────────────────────────────────────────────────────
for contrast_name, (g1, g2) in CONTRASTS.items():
    df_ttest = run_parcelwise_ttest_vol(Y_aligned, design, g1, g2)
    df_ttest_reset = df_ttest.reset_index().rename(columns={"index": "parcel"})
    safe_name = contrast_name.lower().replace(" ", "_")
    outname = f"../../results/{safe_name}_parcelwise_ttest_volume_subcortical.csv"
    df_ttest_reset.to_csv(outname, index=False)
    print(f"Saved: {outname}")
    print(df_ttest.sort_values("pvalue").head(5))

In [ ]:
# ── NiSpace group-level colocalization (Hedges' g) ────────────────────────────
df_all, outputs = run_group_comparisons_vol(
    Y_aligned=Y_aligned, design=design, ref_df=df_reference_selected,
    contrasts=CONTRASTS, labels=GROUP_LABELS, n_perm=N_PERM,
)
df_all.to_csv("../../results/nispace_group_comparison_results_subgroups_subcortical.csv", index=False)
print("Saved group comparison results.")
display(df_all.head(10))

In [ ]:
# ── Single-subject z-score colocalization ─────────────────────────────────────
zscore_outputs = {}
for contrast_name, (gA, gB) in CONTRASTS.items():
    mask = design["CONCOHORT"].astype(float).isin([gA, gB])
    y = Y_aligned.loc[mask].copy()
    d = design.loc[mask].copy()
    design_sub = build_nispace_design_vol(d, gA, gB)
    cov_cols = [c for c in ["groups","age","SEX","eTIV","field_strength"] if c in design_sub.columns]
    # Skip parcels that are all-NaN (VentralDC absent in subject data) before row filter
    valid_parcels = y.columns[~y.isna().all(axis=0)]
    keep = ~(y[valid_parcels].isna().any(axis=1) | design_sub[cov_cols].isna().any(axis=1))
    y          = y.loc[keep]
    d          = d.loc[keep]
    design_sub = design_sub.loc[keep]
    print(f"\n{contrast_name}: {d['CONCOHORT'].astype(float).map(GROUP_LABELS).value_counts().to_dict()}")
    colocs, pvals, qvals, nsp = group_comparison(
        y=y, x=df_reference_selected, parcellation=PARCELLATION, design=design_sub,
        comparison_method="zscore(a,b)", colocalization_method="spearman",
        n_perm=N_PERM, n_proc=-1, verbose=False, plot_design=False,
    )
    zscore_outputs[contrast_name] = {"colocs": colocs, "p": pvals, "q": qvals, "nsp": nsp}

In [ ]:
# ── Build merged dataframe for clinical correlation ───────────────────────────
all_rows = []
for contrast, res in zscore_outputs.items():
    colocs = res["colocs"].copy()
    if colocs.index.name is None:
        colocs.index.name = "PATNO"
    colocs_long = colocs.stack(list(range(colocs.columns.nlevels))).reset_index()
    colocs_long = colocs_long.rename(columns={0: "colocalization"})
    meta_cols = [c for c in colocs_long.columns if c not in ["PATNO", "colocalization"]]
    colocs_long["map"] = colocs_long[meta_cols].astype(str).agg(" | ".join, axis=1)
    colocs_long = colocs_long[["PATNO", "map", "colocalization"]].copy()
    colocs_long["contrast"] = contrast
    all_rows.append(colocs_long)

nispace_df = pd.concat(all_rows, ignore_index=True)
clinical_df = df[["PATNO", "updrs3_score", "moca", "gds", "SEX", "age"]].copy()
nispace_df["PATNO"] = nispace_df["PATNO"].astype(str)
clinical_df["PATNO"] = clinical_df["PATNO"].astype(str)
merged_df = nispace_df.merge(clinical_df, on="PATNO", how="inner")
merged_df.to_csv("../../data/merged_df_volume_subcortical_subgroups.csv", index=False)
print("Saved merged_df_volume_subcortical_subgroups.csv")
print(merged_df.shape)